# Bina 

## Fragestellung

Inwieweit lassen sich Gesundheitskosten in der Schweiz durch demografische (insbesondere Altersstruktur) und regionale Faktoren erklären und welche Segmente verursachen die grösste Systembelastung?

1. Wie entwickeln sich die Gesundheitskosten über die Zeit?
2. Wie unterscheiden sich Kosten nach Alter und Region?
3. Welchen Anteil haben ältere Altersgruppen (65+) an den Gesamtkosten und wie verändert sich dieser über die Zeit?
4. Lassen sich auf Basis von Altersstruktur und Kosten klare Segmente (Cluster) identifizieren?
5. Welche Faktoren erklären die Unterschiede zwischen diesen Segmenten und welche sollten priorisiert werden?

Ziel ist es, datenbasierte Entscheidungsgrundlagen für die Priorisierung von Massnahmen im Gesundheitssystem abzuleiten.

In [ ]:
import pandas as pd 
import numpy as np

## Dataset

[Link](https://www.bfs.admin.ch/asset/de/DF_COU_HEALTH_COSTS)

In [ ]:
df = pd.read_csv("../data/gesuntheitskosten.csv")
df.head()

## Data Details

In [ ]:
print(f'columns: \n {df.columns}')
print(f'shape: \n {df.shape}')


In [ ]:
cols_to_check = [
    "P", "Service provider",
    "S", "Service",
    "M", "Mode of provision",
    "F", "Financing scheme",
    "AGE", "Age groups",
    "GENDER", "Sex",
    "CANTON", "Swiss cantons",
    "TIME_PERIOD",
    "UNIT", "Unit Multiplier",
]

for col in cols_to_check:
    print(f"\n{col}:")
    print(df[col].unique()[:10])

In [ ]:
for series_name, series in df.items():
    print(f'{series_name}: {series.nunique()}')

In [ ]:
df["AGE"].value_counts().head(10)

In [ ]:
df[["OBS_VALUE", "MULT"]].describe()

## Some Data Cleanup

In [ ]:

df_filtered = df[
    (df["P"] == "_T") &
    (df["S"] == "_T") &
    (df["M"] == "_T") &
    (df["F"] == "_T") &
    (df["GENDER"] == "_T")
].copy()

df_filtered = df_filtered[
    [
        "CANTON",
        "Swiss cantons",
        "AGE",
        "Age groups",
        "TIME_PERIOD",
        "OBS_VALUE",
        "MULT",
        "Measurement unit",
        "OBS_STATUS",
        "Code list for Observation Status",
    ]
].copy()

df_filtered = df_filtered.rename(
    columns={
        "CANTON": "canton",
        "Swiss cantons": "canton_name",
        "AGE": "age_code",
        "Age groups": "age_label",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "costs_raw",
        "MULT": "mult",
        "Measurement unit": "measurement_unit",
        "OBS_STATUS": "obs_status",
        "Code list for Observation Status": "obs_status_label",
    }
)

df_filtered["year"] = pd.to_numeric(df_filtered["year"], errors="coerce").astype("Int64")
df_filtered["costs_raw"] = pd.to_numeric(df_filtered["costs_raw"], errors="coerce")
df_filtered["mult"] = pd.to_numeric(df_filtered["mult"], errors="coerce")

df_filtered["costs"] = df_filtered["costs_raw"] * (10 ** df_filtered["mult"])

df_filtered = df_filtered.reset_index(drop=True)

print(df_filtered.shape)
display(df_filtered.head())

### Validation and fixing fails

In [ ]:
check = (
    df_filtered
    .groupby(["canton", "year", "age_code"])
    .size()
)

check.value_counts()

In [ ]:
df_filtered.groupby(
    ["canton", "year", "age_code"]
)["obs_status"].nunique().value_counts()

In [ ]:
df_filtered.columns.tolist()

In [ ]:
cols_to_test = [
    'canton',
    'canton_name',
    'age_code',
    'age_label',
    'year',
    'costs_raw',
    'mult',
    'measurement_unit',
    'obs_status',
    'obs_status_label',
    'costs'
 ]

for col in cols_to_test:
    test = (
        df_filtered
        .groupby(["canton", "year", "age_code"])[col]
        .nunique()
        .value_counts()
    )
    print(f"\n{col}:")
    print(test)

In [ ]:
df_filtered["measurement_unit"].value_counts()

In [ ]:
df_filtered = df_filtered[
    df_filtered["measurement_unit"] == "Swiss franc"
].copy()

In [ ]:
df_filtered.groupby(["canton", "year", "age_code"]).size().value_counts()

## Some more Data cleaning

In [ ]:
df_detail = df_filtered[
    (df_filtered["canton"] != "_T") &
    (df_filtered["age_code"] != "_T")
].copy()

df_canton_total = df_filtered[
    (df_filtered["canton"] != "_T") &
    (df_filtered["age_code"] == "_T")
].copy()

df_total = df_filtered[
    (df_filtered["canton"] == "_T") &
    (df_filtered["age_code"] == "_T")
].copy()

age_order = [
    "0-5 years", "6-10 years", "11-15 years", "16-20 years",
    "21-25 years", "26-30 years", "31-35 years", "36-40 years",
    "41-45 years", "46-50 years", "51-55 years", "56-60 years",
    "61-65 years", "66-70 years", "71-75 years", "76-80 years",
    "81-85 years", "86-90 years", "91-95 years", "96 years or older"
]

## First Plots

In [ ]:
import plotly.io as pio
pio.renderers.default = "browser"

In [ ]:
import plotly.express as px

px.line(
    df_total,
    x="year",
    y="costs",
    title="Gesamte Gesundheitskosten Schweiz"
)

In [ ]:
df_age = (
    df_detail
    .groupby("age_label", as_index=False)["costs"]
    .mean()
)

px.bar(
    df_age,
    x="age_label",
    y="costs",
    category_orders={"age_label": age_order},
    title="Kosten nach Altersgruppe"
)

In [ ]:
# TODO: Order Age correctly

pivot = df_detail.pivot_table(
    index="age_label",
    columns="year",
    values="costs",
    aggfunc="mean",
)

px.imshow(pivot, aspect="auto",
          title="Kosten nach Jahr und Altersgruppe",     
)

In [ ]:
df_canton = (
    df_canton_total
    .groupby("canton_name", as_index=False)["costs"]
    .mean()
    .sort_values("costs", ascending=False)
)

px.bar(df_canton.head(10), x="canton_name", y="costs", title="Kosten pro Kanton (Top 10)")

In [ ]:
df_age_total = (
    df_detail
    .groupby("age_label", as_index=False)["costs"]
    .sum()
)

In [ ]:
df_age_total["share"] = df_age_total["costs"] / df_age_total["costs"].sum()

In [ ]:
px.bar(df_age_total, x="age_label", 
       y="share", category_orders={"age_label": age_order}, 
       title="Anteil der Kosten pro Altergruppe (%)")

In [ ]:
df_age_time = (
    df_detail
    .groupby(["year", "age_label"])["costs"]
    .sum()
    .reset_index()
)

In [ ]:
px.line(df_age_time, x="year", y="costs", color="age_label", category_orders={"age_label": age_order})

In [ ]:
df_detail["age_start"] = df_detail["age_label"].str.extract(r"(\d+)").astype(int)
df_detail["is_65_plus"] = df_detail["age_start"] >= 65

df_65 = (
    df_detail
    .groupby(["year", "is_65_plus"])["costs"]
    .sum()
    .reset_index()
)
df_65_total = df_65.groupby("year")["costs"].transform("sum")
df_65["share"] = df_65["costs"] / df_65_total

In [ ]:
df_65_plus = df_65[df_65["is_65_plus"]].copy()
df_65_plus["share_pct"] = df_65_plus["share"] * 100

px.line(
    df_65_plus,
    x="year",
    y="share_pct",
    title="Anteil der Gesundheitskosten durch 65+ (%)"
)

### Validation to explain dip

There is a weird dip in 2021. Check to see if it's a fail or maybe Corona

In [ ]:
df_total_manual = (
    df_detail
    .groupby("year", as_index=False)["costs"]
    .sum()
    .rename(columns={"costs": "costs_manual"})
)

df_total_official = (
    df_total[["year", "costs"]]
    .rename(columns={"costs": "costs_official"})
)

df_compare = df_total_manual.merge(
    df_total_official,
    on="year",
    how="inner"
)

df_compare["diff"] = df_compare["costs_manual"] - df_compare["costs_official"]
df_compare["rel_diff_pct"] = df_compare["diff"] / df_compare["costs_official"] * 100

df_compare

In [ ]:
df_total["costs"]

## Additional Dataset

We need info about the population, to make sure the where the real cost is. There are fewer 96+ people -> cost per person could be higher

[Link](https://www.bfs.admin.ch/bfs/de/home/statistiken/bevoelkerung.assetdetail.36074768.html)

In [ ]:
df_demographics = pd.read_csv("../data/px-x-0102020000_104_20260410-181954.csv", encoding="latin1")
df_demographics.head()

In [ ]:
df_demographics["Kanton"].value_counts()

In [ ]:
df_premiums = pd.read_csv("../data/Prämien_CH.csv")

In [ ]:
df_premiums["Geschäftsjahr"].unique()

## Add some Machine learning

Time for more cleaning, one hot encoding, feature engineering etc

- K-Means for groups
- RandoForest for feature importance